In [9]:
import pandas as pd
import numpy as np

In [10]:
fp = 'attachments/geochemical_data.xlsx'

df_raw = pd.read_excel(fp)

# rename cols
df_clean = df_raw.rename(
    columns={
        df_raw.columns[1]: "EL_wt%",
        df_raw.columns[2]: "CI_wt%",
        "oxide component": "oxide",
        df_raw.columns[4]: "BSE_Sun_McDonough_wt%",
    }
)

# elemental abundances (el & ci)
df_elements = df_clean[["element", "EL_wt%", "CI_wt%"]]

# oxide reference abundances (mcdonough & sun 1995)
df_oxides = df_clean[["oxide", "BSE_Sun_McDonough_wt%"]].dropna()

display(df_elements)
display(df_oxides)

,element,EL_wt%,CI_wt%
0,Na,1.08,1.14
1,Mg,22.30,21.90
2,Al,1.68,1.94
3,Si,32.60,24.40
4,S,3.10,3.19
5,K,0.08,0.12
6,Ca,1.52,2.13
7,Ti,0.08,0.10
8,Cr,0.35,0.60
9,Fe,35.20,42.00


,oxide,BSE_Sun_McDonough_wt%
0,Na2O,0.360
1,MgO,37.800
2,Al2O3,4.450
3,SiO2,45.000
4,K2O,0.029
5,CaO,3.550
6,TiO2,0.200
7,Cr2O3,0.384
8,FeO,8.050
9,NiO,0.250


In [11]:
# atomic weights (g/mol)
atomic_weights = {
    "Na": 22.9897,
    "Mg": 24.305,
    "Al": 26.9815,
    "Si": 28.0855,
    "S": 32.06,
    "K": 39.0983,
    "Ca": 40.078,
    "Ti": 47.867,
    "Cr": 51.9961,
    "Fe": 55.845,
    "Ni": 58.6934,
    "O": 15.999,
}

# oxide stoichometry map: (cations, oxygen atoms)
oxide_stoich = {
    "Na": ("Na2O", 2, 1),
    "Mg": ("MgO", 1, 1),
    "Al": ("Al2O3", 2, 3),
    "Si": ("SiO2", 1, 2),
    "K": ("K2O", 2, 1),
    "Ca": ("CaO", 1, 1),
    "Ti": ("TiO2", 1, 2),
    "Cr": ("Cr2O3", 2, 3),
    "Fe": ("FeO", 1, 1),
    "Ni": ("NiO", 1, 1),
}

# Calculate oxide conversion factors
conversion_factors = {}
for elem, (ox_name, n_cat, n_ox) in oxide_stoich.items():
    mol_wt_oxide = n_cat * atomic_weights[elem] + n_ox * atomic_weights["O"]
    conversion_factors[elem] = mol_wt_oxide / (n_cat * atomic_weights[elem])

df_factors = pd.DataFrame(
    list(conversion_factors.items()), columns=["element", "conversion_factor"]
)
df_factors

,element,conversion_factor
0,Na,1.347960
1,Mg,1.658260
2,Al,1.889443
3,Si,2.139307
4,K,1.204600
5,Ca,1.399197
6,Ti,1.668477
7,Cr,1.461544
8,Fe,1.286489
9,Ni,1.272586


## Question 5A

In [12]:
X_core = 0.30  # mass fraction of core
X_bse = 0.70  # mass fraction of BSE

In [13]:
# map element to oxide name
elem_to_oxide = {elem: stoich[0] for elem, stoich in oxide_stoich.items()}

# set up reference oxides lookup dictionary
bse_ref = dict(zip(df_oxides["oxide"], df_oxides["BSE_Sun_McDonough_wt%"]))

In [14]:
def get_calculated_bse(f_EL):
    f_CI = 1.0 - f_EL

    # bulk comp calc
    bulk_comp = {}
    for elem in df_elements["element"]:
        el_val = df_elements.loc[
            df_elements["element"] == elem, "EL_wt%"
        ].values[0]
        ci_val = df_elements.loc[
            df_elements["element"] == elem, "CI_wt%"
        ].values[0]
        bulk_comp[elem] = f_EL * el_val + f_CI * ci_val

    # put siderophile elements (Fe, Ni, S) into core (30 wt%)
    siderophile_bulk = bulk_comp["Fe"] + bulk_comp["Ni"]

    # Fe and Ni core mass fractions (rel to whole planet)
    fe_in_core = (bulk_comp["Fe"] / siderophile_bulk) * (
        X_core * 100 - bulk_comp["S"]
    )
    ni_in_core = (bulk_comp["Ni"] / siderophile_bulk) * (
        X_core * 100 - bulk_comp["S"]
    )

    # put elements in silicate reservoir (BSE)
    silicate_elem = {}
    for elem, val in bulk_comp.items():
        if elem == "S":
            continue  # S stays completely in the core
        elif elem == "Fe":
            silicate_elem[elem] = max(0.0, (val - fe_in_core) / X_bse)
        elif elem == "Ni":
            silicate_elem[elem] = max(0.0, (val - ni_in_core) / X_bse)
        else:
            # Lithophile elements stay 100% in BSE
            silicate_elem[elem] = val / X_bse

    # convert elements in BSE to oxide wt%s
    silicate_oxides = {}
    for elem, val in silicate_elem.items():
        if elem in conversion_factors:
            ox_name = elem_to_oxide[elem]
            silicate_oxides[ox_name] = val * conversion_factors[elem]

    # renormalize silicate oxides to 100%
    total_wt = sum(silicate_oxides.values())
    renorm_oxides = {
        ox: (wt / total_wt) * 100.0 for ox, wt in silicate_oxides.items()
    }

    df_res = pd.DataFrame(
        list(renorm_oxides.items()), columns=["oxide", "calculated_BSE_wt%"]
    )
    return df_res

# sum of sq err relative to Sun & McDonough (1995)
def calculate_sse(f_EL):
    df_calc = get_calculated_bse(f_EL)

    # Merge calculated values with reference BSE values
    df_merged = pd.merge(df_oxides, df_calc, on="oxide")

    # SSE relative to Sun & McDonough (1995)
    sse = (
        (df_merged["calculated_BSE_wt%"] - df_merged["BSE_Sun_McDonough_wt%"])
        ** 2
    ).sum()
    return sse

In [15]:
# grid search for best fit
f_grid = np.linspace(0.0, 1.0, 1001)
sse_grid = np.array([calculate_sse(f) for f in f_grid])

# find the minimum SSE
best_idx = np.argmin(sse_grid)
opt_f_EL = f_grid[best_idx]
min_sse = sse_grid[best_idx]

print(f"Optimal EL Fraction (f_EL): {opt_f_EL:.4f}")
print(f"Optimal CI Fraction (f_CI): {1 - opt_f_EL:.4f}")
print(f"Minimum SSE: {min_sse:.6f}")

Optimal EL Fraction (f_EL): 0.4320
Optimal CI Fraction (f_CI): 0.5680
Minimum SSE: 122.757382


## Question 5B

In [21]:
# compare to sun & mcdonough (1995) reference values
df_comp = pd.merge(df_oxides, get_calculated_bse(opt_f_EL), on="oxide")
df_comp["absolute_deviation"] = (
    df_comp["calculated_BSE_wt%"] - df_comp["BSE_Sun_McDonough_wt%"]
).abs()
df_comp["relative_error_%"] = (
    df_comp["absolute_deviation"] / df_comp["BSE_Sun_McDonough_wt%"]
) * 100

display(df_comp.sort_values("relative_error_%", ascending=False))

,oxide,BSE_Sun_McDonough_wt%,calculated_BSE_wt%,absolute_deviation,relative_error_%
4,K2O,0.029,0.100127,0.071127,245.264251
0,Na2O,0.360,1.215193,0.855193,237.553637
9,NiO,0.250,0.824245,0.574245,229.697991
8,FeO,8.050,14.257757,6.207757,77.114995
7,Cr2O3,0.384,0.581874,0.197874,51.529676
5,CaO,3.550,2.113265,1.436735,40.471413
6,TiO2,0.200,0.123347,0.076653,38.326551
2,Al2O3,4.450,2.794382,1.655618,37.204902
1,MgO,37.800,29.618415,8.181585,21.644406
3,SiO2,45.000,48.371396,3.371396,7.491991


In [24]:
# wt by reference concentration to prevent minor elements from dominating err
df_comp["weighted_sq_err"] = (
    (df_comp["calculated_BSE_wt%"] - df_comp["BSE_Sun_McDonough_wt%"])
    / df_comp["BSE_Sun_McDonough_wt%"]
) ** 2

weighted_sse = df_comp["weighted_sq_err"].sum()

display(df_comp.sort_values("weighted_sq_err", ascending=False))

,oxide,BSE_Sun_McDonough_wt%,calculated_BSE_wt%,absolute_deviation,relative_error_%,weighted_sq_err
4,K2O,0.029,0.100127,0.071127,245.264251,6.015455
0,Na2O,0.360,1.215193,0.855193,237.553637,5.643173
9,NiO,0.250,0.824245,0.574245,229.697991,5.276117
8,FeO,8.050,14.257757,6.207757,77.114995,0.594672
7,Cr2O3,0.384,0.581874,0.197874,51.529676,0.265531
5,CaO,3.550,2.113265,1.436735,40.471413,0.163794
6,TiO2,0.200,0.123347,0.076653,38.326551,0.146892
2,Al2O3,4.450,2.794382,1.655618,37.204902,0.138420
1,MgO,37.800,29.618415,8.181585,21.644406,0.046848
3,SiO2,45.000,48.371396,3.371396,7.491991,0.005613


Relative to McDonough and Sun (1995), the simple EL/CI fit is not a match. The element ratios for the optimal EL/CI composition of K2O, Na2O, and NiO are the most problematic, based on both the relative percent error and the weighted error (to balance the effect of the trace elements). This simple mixing model assumes that 100% of the volatiles are retained during accretion, but the CI composition has a higher proportion of volatiles than the Earth actually has, so volatiles like K2O and Na2O are overestimated. NiO is likely overestimated because it does not account for the fact that Ni prefers to be in the metallic phase, and more would partition into the core. 

## Question 5C 

This simple mixing model makes several assumptions, including that the composition can be composed of just CI and EL chondrites; however, other chondrites may also play a role. As shown in the models of Alexander (2022), using parts of the chondrite composition instead of the whole composition may allow a better fit, especially for volatile elements that can be better accounted for by chondrule compositions. 

If the Earth was built of EL components, which are very reduced, it would suggest that the accretion happened in an oxygen-poor environment. 

This model also does not account for the change in Earth's size and pressure/temperature conditions over time, which would change how different phases partition into the core vs. the silicate component of the Earth. Other events that could change the Earth's composition are big collision events, like the moon colliding with Earth. 